# Шаг 1. Данные и подготовка текстов

Задача: по вопросу пользователя в поддержку Авито найти и отранжировать статьи справки.

Пара слов о том, куда это встраивается. RAG (Retrieval-Augmented Generation) - это схема,
в которой языковая модель отвечает пользователю, опираясь на найденные
документы. Пайплайн состоит из двух больших частей: сначала **retrieval** (поиск подходящих
документов по запросу), потом генерация ответа на их основе. В этой задаче генерации нет
вообще - нужно решить только поисковую часть: вернуть для каждого запроса топ-10 статей,
отсортированных по убыванию релевантности. Качество меряется метрикой MAP@10 (подробно
разберу её в шаге 2).

В этом ноутбуке я:

1. загружаю три исходных файла и проверяю их на аномалии;
2. смотрю глазами на запросы и статьи - без этого непонятно, с каким шумом придётся работать;
3. чищу HTML статей до связного текста;
4. режу статьи на чанки - фрагменты фиксированной длины в токенах - для векторного поиска;
5. сохраняю подготовленные данные на Google Диск для следующих шагов.

Запуск: обычный CPU-runtime в Google Colab, примерно 10-15 минут.

## 0. Установка зависимостей

In [1]:
import importlib.util
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--upgrade-strategy", "only-if-needed",
        "pandas", "pyarrow", "beautifulsoup4", "lxml",
        "transformers>=4.50,<5", "tqdm",
    ])
    print("Зависимости установлены.")
else:
    print("Не Colab: ставлю на то, что зависимости уже есть (см. requirements.txt).")

Зависимости установлены.


## 1. Google Диск и структура проекта

Весь проект живёт в одной папке на Google Диске:

```text
MyDrive/
└── avito_rag/
    ├── data/                  ← сюда НУЖНО положить три файла задачи
    │   ├── articles.f
    │   ├── calibration.f
    │   └── test.f
    ├── artifacts/             ← создаётся ноутбуками автоматически
    └── answer.csv             ← итоговый файл, появится после шага 4
```

От вас требуется ровно одно действие: создать на Диске папку `avito_rag/data` и положить
в неё три исходных файла. Всё остальное - папку `artifacts`, промежуточные файлы, итоговый
`answer.csv` - ноутбуки создают и перезаписывают сами. Повторный запуск безопасен:
тяжёлые артефакты кэшируются и пересчитываются только при изменении входных данных.

In [3]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/avito_rag")
else:
    # Локальный запуск для отладки: папка avito_rag рядом с ноутбуками.
    PROJECT_ROOT = Path.cwd().resolve().parent / "avito_rag"

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

missing = [
    str(DATA_DIR / name)
    for name in ("articles.f", "calibration.f", "test.f")
    if not (DATA_DIR / name).is_file()
]
if missing:
    raise FileNotFoundError(
        "Не найдены исходные файлы задачи. Создайте папку avito_rag/data "
        "на Google Диске и положите туда articles.f, calibration.f, test.f.\n"
        "Отсутствуют:\n" + "\n".join(missing)
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Исходные данные на месте.")

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/avito_rag
Исходные данные на месте.


## 2. Загрузка и первичные проверки

Файлы в формате Feather (колоночный бинарный формат, читается pandas напрямую).
Прежде чем что-то строить, я проверяю базовые вещи: уникальность идентификаторов,
пропуски, дубликаты.

In [4]:
import re
import unicodedata
from collections import Counter
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 60)

articles = pd.read_feather(DATA_DIR / "articles.f")
calibration = pd.read_feather(DATA_DIR / "calibration.f")
test = pd.read_feather(DATA_DIR / "test.f")

print("articles:", articles.shape)
print("calibration:", calibration.shape)
print("test:", test.shape)

assert articles["article_id"].is_unique
assert calibration["query_id"].is_unique
assert test["query_id"].is_unique
assert articles["title"].notna().all() and articles["body"].notna().all()
assert calibration["query_text"].astype(str).str.strip().ne("").all()
assert test["query_text"].astype(str).str.strip().ne("").all()

display(articles.head(2))
display(calibration.head(3))
display(test.head(3))

/usr/local/lib/python3.12/dist-packages/pandas/io/feather_format.py:178: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  pa_table = feather.read_table(


articles: (793, 3)
calibration: (500, 3)
test: (500, 2)


,article_id,title,body
0,1730,Имя или название компании,"<ol><li><p>Зайдите в раздел <a href=""https://www.avito.ru/profile/extended"">Управление профилем</a>.</p><ol start=""2..."
1,1746,"Понять, что профиль заблокирован","<p>Проверьте, какое сообщение вы видите при входе в профиль.</p><div class=""tabset tabset_155""><input type=""radio"" n..."


,query_id,query_text,ground_truth
0,1,Как передать товар через службу авито,1909 4234
1,2,"Можете подсказать, если заказать товар Авито доставкой и оплатить также через авито, в случае возврата товара деньги...",2865 4400
2,3,Здравствуйте. Как отправить товар через Авито.,1909


,query_id,query_text
0,1,"Здравствуйте! Подскажите, пожалуйста, не могу оформить доставку , какие-то проблемы с приложением?"
1,2,"Здравствуйте , почему так долго доставляется возврат? В приложении СДЭКА пишет то что посылка вручена, а на Авито до..."
2,3,"Здравствуйте,подскажите как мне отправить кроссовки покупателю через пункт «Авито»?"


## 3. Разбор разметки `ground_truth`

В `calibration.f` правильные статьи записаны строкой вида `"1909 4396"`. Я аккуратно
разбираю её в список целых `article_id` и отдельно фиксирую аномалии: нечисловые токены,
повторы внутри одной строки, ссылки на несуществующие статьи. Заодно смотрю, сколько
релевантных статей приходится на запрос - от этого зависит, насколько важно уметь
находить несколько документов, а не один лучший.

In [5]:
def parse_ground_truth_value(value: Any) -> tuple[list[int], list[str]]:
    anomalies: list[str] = []
    if pd.isna(value):
        anomalies.append("missing")
        return [], anomalies
    text = str(value).strip()
    if not text:
        return [], anomalies
    ids: list[int] = []
    seen: set[int] = set()
    for token in re.split(r"\s+", text):
        if not re.fullmatch(r"\d+", token):
            anomalies.append(f"non_integer_token={token!r}")
            continue
        article_id = int(token)
        if article_id in seen:
            anomalies.append(f"duplicate_id={article_id}")
            continue
        seen.add(article_id)
        ids.append(article_id)
    return ids, anomalies


def parse_ground_truth_series(series: pd.Series) -> tuple[pd.Series, pd.DataFrame]:
    parsed, anomaly_rows = [], []
    for idx, value in series.items():
        ids, anomalies = parse_ground_truth_value(value)
        parsed.append(ids)
        for anomaly in anomalies:
            anomaly_rows.append({"row_index": idx, "raw_ground_truth": value, "anomaly": anomaly})
    return pd.Series(parsed, index=series.index, name="ground_truth_ids"), pd.DataFrame(anomaly_rows)


def soft_normalize_text(value: Any) -> str:
    """Мягкая нормализация: NFKC-юникод, неразрывные пробелы, схлопывание пробелов.

    Текст по смыслу не меняется — это нужно только чтобы сравнивать строки между собой.
    """
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value)).replace("\u00A0", " ")
    return re.sub(r"\s+", " ", text).strip()


calibration = calibration.copy()
calibration["ground_truth_ids"], gt_anomalies = parse_ground_truth_series(calibration["ground_truth"])
calibration["ground_truth_count"] = calibration["ground_truth_ids"].map(len)

article_id_set = set(articles["article_id"].tolist())
bad_refs = calibration["ground_truth_ids"].map(lambda ids: [x for x in ids if x not in article_id_set])

print("Аномалий парсинга:", len(gt_anomalies))
print("Ссылок на несуществующие статьи:", int(bad_refs.map(bool).sum()))
assert calibration["ground_truth_count"].gt(0).all(), "Есть запросы без разметки"

print("\nСколько релевантных статей приходится на один запрос:")
display(calibration["ground_truth_count"].value_counts().sort_index().to_frame("запросов"))

Аномалий парсинга: 0
Ссылок на несуществующие статьи: 0

Сколько релевантных статей приходится на один запрос:


,запросов
ground_truth_count,
1,279
2,182
3,38
4,1


Больше чем у трети запросов релевантных статей две и более. Это важное наблюдение:
решение, которое идеально находит одну лучшую статью, но игнорирует остальные,
упрётся в потолок по MAP@10. Значит, кандидатов нужно набирать с запасом и ранжировать
весь список, а не искать один ответ.

## 4. Какие бывают запросы

Смотрю на длины и на сами тексты. Запросы - живые сообщения в поддержку: приветствия,
опечатки, слитая пунктуация, разговорные формулировки. Язык запросов заметно отличается
от канцелярского языка статей - это главный источник сложности задачи: чисто словарное
совпадение часто не срабатывает.

In [6]:
for name, frame in [("calibration", calibration), ("test", test)]:
    lengths = frame["query_text"].astype(str).str.len()
    words = frame["query_text"].map(lambda x: len(re.findall(r"\w+", str(x), flags=re.UNICODE)))
    print(
        f"{name}: длина в символах медиана={lengths.median():.0f} (p95={lengths.quantile(0.95):.0f}), "
        f"в словах медиана={words.median():.0f}"
    )

print("\nПримеры коротких запросов:")
short = calibration.assign(w=calibration["query_text"].map(lambda x: len(str(x).split()))).nsmallest(5, "w")
display(short[["query_id", "query_text"]])

print("Примеры запросов с «человеческим» шумом:")
noisy = calibration[calibration["query_text"].str.contains(r"[,.!?][А-Яа-яA-Za-z]|здравствуйте|подскажите", case=False, regex=True)]
display(noisy[["query_id", "query_text"]].head(5))

calibration: длина в символах медиана=69 (p95=140), в словах медиана=10
test: длина в символах медиана=66 (p95=152), в словах медиана=10

Примеры коротких запросов:


,query_id,query_text
53,54,"Здравствуйте,оплатила объявление"
76,77,Здравствуйте.как отменить аукцион?
82,83,"Здравствуйте, я оплатила объявление"
232,233,"здравствуйте,почему такая дорогая доставка"
326,327,Как воспользоваться Авито Доставкой?


Примеры запросов с «человеческим» шумом:


,query_id,query_text
2,3,Здравствуйте. Как отправить товар через Авито.
9,10,"хочу отменить заказ,мне вернут полную сумму?"
14,15,"Здравствуйте! Меня в пункте выдачи ждет заказ. Если я его не забираю, то мне деньги вернуться на карту?"
17,18,"Здравствуйте, почему сумма доставки меняется при оформлении?"
22,23,Здравствуйте! Почему нельзя списать все баллы на доставку в данном объявлении? Пишет что доступно к списанию только ...


## 5. Очистка HTML статей

Тела статей - это HTML с таблицами, вкладками, спойлерами и служебной разметкой.
Модели нужен связный текст, поэтому я разбираю HTML парсером (BeautifulSoup + lxml)
и аккуратно превращаю разметку в текст. Мои принципы:

- **удаляю** только технические элементы без видимого текста: `script`, `style`, `noscript`, `svg`, `input`, `iframe`;
- **сохраняю** всё, что пользователь видит на странице: заголовки и абзацы отдельными строками,
  пункты списков с маркером `- `, строки таблиц через ` | `, текст внутри вкладок и спойлеров,
  видимый текст ссылок (сами URL из `href` не тащу - это шум для поиска);
- подписи к картинкам (`alt`) оставляю только когда это осмысленное описание, а не имя файла
  или URL - для этого отдельный фильтр с декодированием percent-encoding;
- если первая строка тела дублирует заголовок, убираю её, чтобы заголовок не считался дважды.

Из очищенного заголовка и тела собираю `document_text` - единый текст статьи с явной
структурой «Заголовок: … / Текст статьи: …».

In [7]:
import html
from urllib.parse import unquote

from bs4 import BeautifulSoup, NavigableString

TECHNICAL_TAGS_TO_REMOVE = ["script", "style", "noscript", "svg", "input", "iframe"]
BLOCK_TAGS = ["p", "div", "section", "article", "header", "footer", "aside", "blockquote", "pre", "details", "summary"]
HEADING_TAGS = ["h1", "h2", "h3", "h4", "h5", "h6"]
IMAGE_EXTENSIONS = "png|jpe?g|gif|svg|webp|bmp|tiff?|ico|avif"


def normalize_line(text: str) -> str:
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text).replace("\u00A0", " ")
    return re.sub(r"[ \t\r\f\v]+", " ", text).strip()


def collapse_blank_lines(text: str, max_blank: int = 1) -> str:
    lines = [normalize_line(line) for line in text.splitlines()]
    result, blank_count = [], 0
    for line in lines:
        if not line:
            blank_count += 1
            if blank_count <= max_blank and result:
                result.append("")
            continue
        blank_count = 0
        result.append(line)
    return "\n".join(result).strip()


def table_to_text(table_tag) -> str:
    rows = []
    for tr in table_tag.find_all("tr"):
        cells = [normalize_line(cell.get_text(" ", strip=True)) for cell in tr.find_all(["th", "td"])]
        cells = [cell for cell in cells if cell]
        if cells:
            rows.append(" | ".join(cells))
    return "\n".join(rows)


def assess_image_alt(value: Any) -> tuple[bool, str, str]:
    """Решаю, полезна ли подпись картинки: (оставить?, текст, причина)."""
    if value is None or pd.isna(value):
        return False, "", "empty"
    decoded = normalize_line(unquote(str(value)))
    if not decoded:
        return False, decoded, "empty"
    if re.search(r"(?i)(?:https?://|www\.|data:image/|^//\S+)", decoded):
        return False, decoded, "url"
    if re.search(rf"(?i)(?:^|[/\\])[^/\\]+\.(?:{IMAGE_EXTENSIONS})(?:[?#].*)?$", decoded) or re.fullmatch(
        rf"(?i)[\w .()@+-]+\.(?:{IMAGE_EXTENSIONS})", decoded
    ):
        return False, decoded, "file_name"

    letters = sum(ch.isalpha() for ch in decoded)
    technical = sum(not (ch.isalnum() or ch.isspace() or ch in ".,:;!?—–-()«»№%+/\"") for ch in decoded)
    if technical / max(len(decoded), 1) > 0.40 and letters < max(6, len(decoded) * 0.35):
        return False, decoded, "mostly_technical_symbols"

    words = re.findall(r"[A-Za-zА-Яа-яЁё0-9]+", decoded)
    has_cyrillic = bool(re.search(r"[А-Яа-яЁё]", decoded))
    compact_ascii_id = (
        not has_cyrillic
        and len(decoded) <= 40
        and bool(re.fullmatch(r"[A-Za-z0-9_.-]+", decoded))
        and ("_" in decoded or "-" in decoded or len(words) <= 1)
    )
    if compact_ascii_id:
        return False, decoded, "short_technical_identifier"
    if letters < 3 or len(decoded) < 4:
        return False, decoded, "too_short"
    if not has_cyrillic and len(words) == 1 and len(decoded) <= 24:
        return False, decoded, "short_technical_identifier"
    return True, decoded, "meaningful_description"


def preserve_image_alt(soup: BeautifulSoup) -> None:
    for img in soup.find_all("img"):
        useful, alt, _ = assess_image_alt(img.get("alt"))
        if useful:
            img.replace_with(NavigableString(f"\nИзображение: {alt}\n"))
        else:
            img.decompose()


def clean_article_html(html_value: Any) -> str:
    if pd.isna(html_value):
        return ""
    html_text = str(html_value).strip()
    if not html_text:
        return ""
    soup = BeautifulSoup(html.unescape(html_text), "lxml")
    for tag in soup(TECHNICAL_TAGS_TO_REMOVE):
        tag.decompose()
    preserve_image_alt(soup)
    for table in soup.find_all("table"):
        table_text = table_to_text(table)
        table.replace_with(NavigableString("\n" + table_text + "\n" if table_text else "\n"))
    for br in soup.find_all("br"):
        br.replace_with(NavigableString("\n"))
    for tag in soup.find_all(HEADING_TAGS + BLOCK_TAGS):
        tag.insert_before(NavigableString("\n"))
        tag.append(NavigableString("\n"))
    for li in soup.find_all("li"):
        li.insert_before(NavigableString("\n- "))
        li.append(NavigableString("\n"))
    cleaned = soup.get_text(" ")
    cleaned = re.sub(r"\n\s*", "\n", cleaned)
    cleaned = re.sub(r"\s*\n", "\n", cleaned)
    cleaned = re.sub(r"(?m)^-\s+", "- ", cleaned)
    return collapse_blank_lines(cleaned, max_blank=1)


def clean_title(value: Any) -> str:
    return "" if pd.isna(value) else normalize_line(str(value))


def remove_duplicate_leading_title(title: str, body: str) -> str:
    title_norm = soft_normalize_text(title).casefold()
    for i, line in enumerate(body.splitlines()):
        if not line.strip():
            continue
        return "\n".join(body.splitlines()[i + 1:]).strip() if soft_normalize_text(line).casefold() == title_norm else body.strip()
    return body.strip()


def make_document_text(title: str, body: str) -> str:
    body_wo_title = remove_duplicate_leading_title(title, body)
    parts = [f"Заголовок: {title}".strip(), "", "Текст статьи:"]
    if body_wo_title:
        parts.append(body_wo_title)
    return "\n".join(parts).strip()


articles_processed = articles.copy()
articles_processed["raw_title"] = articles_processed["title"]
articles_processed["raw_body"] = articles_processed["body"]
articles_processed["clean_title"] = articles_processed["raw_title"].map(clean_title)
articles_processed["clean_body"] = articles_processed["raw_body"].map(clean_article_html)
articles_processed["document_text"] = articles_processed.apply(
    lambda r: make_document_text(r["clean_title"], r["clean_body"]), axis=1
)

print("Очистка завершена для", len(articles_processed), "статей.")

Очистка завершена для 793 статей.


### Контроль качества очистки

Правильность очистки я проверяю двумя способами: глазами на разных типах статей
(обычная, со списком, с таблицей) и автоматической диагностикой на всём корпусе -
не осталось ли HTML-тегов, не исчез ли текст целиком, не появились ли пустые статьи.

In [8]:
def truncate_text(text: Any, limit: int = 700) -> str:
    text = "" if pd.isna(text) else str(text)
    return text if len(text) <= limit else text[:limit] + f"\n... <обрезано, всего {len(text)} символов>"


def contains_html_fragment(text: str) -> bool:
    return bool(re.search(r"</?[a-zA-Z][^>]{0,120}>", str(text)))


# Три примера «до/после»: обычная статья, со списком, с таблицей.
example_masks = [
    ("Обычная статья", articles_processed["raw_body"].str.contains("<p", na=False)
     & ~articles_processed["raw_body"].str.contains("<table|<li", na=False, regex=True)),
    ("Статья со списком", articles_processed["raw_body"].str.contains("<li", na=False)),
    ("Статья с таблицей", articles_processed["raw_body"].str.contains("<table", na=False)),
]
for label, mask in example_masks:
    row = articles_processed[mask].iloc[0]
    print("=" * 100)
    print(f"{label}: article_id={row['article_id']} — {row['clean_title']}")
    print("--- исходный HTML ---")
    print(truncate_text(row["raw_body"], 400))
    print("--- очищенный текст ---")
    print(truncate_text(row["clean_body"], 400))

word_count = articles_processed["clean_body"].map(lambda x: len(re.findall(r"\w+", str(x), flags=re.UNICODE)))
diagnostics = {
    "пустых после очистки": int(articles_processed["clean_body"].str.strip().eq("").sum()),
    "очень коротких (<10 слов)": int(word_count.lt(10).sum()),
    "с остатками HTML-тегов": int(articles_processed["clean_body"].map(contains_html_fragment).sum()),
}
print("\nДиагностика по корпусу:", diagnostics)
assert diagnostics["пустых после очистки"] == 0

Обычная статья: article_id=1784 — Профиль удалён
--- исходный HTML ---
<p>Если профиль был удалён, появится сообщение, что телефон или почта не привязаны к профилю.</p><p><strong>Что делать:</strong> восстановить профиль не получится — даже через поддержку. Вы можете использовать другой свой аккаунт или <a href="https://support.avito.ru/articles/1811">создать новый.</a></p><div class="factoid factoid_warning"><p>Обычно мы удаляем профиль только по просьбе пользовател
... <обрезано, всего 710 символов>
--- очищенный текст ---
Если профиль был удалён, появится сообщение, что телефон или почта не привязаны к профилю.
Что делать: восстановить профиль не получится — даже через поддержку. Вы можете использовать другой свой аккаунт или создать новый.
Обычно мы удаляем профиль только по просьбе пользователя, или если аккаунтом не пользовались в течение 3 лет. Если аккаунт долго был неактивен, мы присылаем письмо, чтобы вы мог
... <обрезано, всего 489 символов>
Статья со списком: article_id=173

## 6. Чанкинг: режу статьи на фрагменты

Дальше (шаг 2) статьи будут искаться векторной моделью `BAAI/bge-m3`. Токен - это единица,
которой модель считает текст (примерно слово или кусок слова). Формально bge-m3 умеет читать
до 8 тысяч токенов, но на длинных текстах эмбеддинг «размывается»: вектор всей статьи хуже
отражает конкретный вопрос, чем вектор релевантного фрагмента. Поэтому я делю тело статьи
на **чанки** - перекрывающиеся окна фиксированной длины.

Параметры - **384 токена с перекрытием 58 (~15%)** - я выбирал на шаге 2 по качеству поиска
на размеченных запросах: сравнивал целые документы против чанков разных размеров, чанки
выигрывали заметно (подробности в README). Перекрытие нужно, чтобы ответ, попавший на границу
двух окон, не потерялся.

Два инженерных момента, на которых легко споткнуться:

- в каждый чанк я добавляю префикс с заголовком статьи - иначе фрагмент из середины
  «Как оформить возврат» теряет контекст, о чём вообще статья;
- окна строю по токенам (кодирую текст токенизатором, режу список токенов, декодирую
  обратно), а не по символам - так гарантируется точный размер и точное перекрытие.

In [ ]:
from transformers import AutoTokenizer

EMBEDDER_NAME = "BAAI/bge-m3"
CHUNK_SIZE = 384
CHUNK_OVERLAP = 58

tokenizer = AutoTokenizer.from_pretrained(EMBEDDER_NAME, use_fast=True)
print("Токенизатор загружен:", tokenizer.__class__.__name__)


def count_model_tokens(text: Any, add_special_tokens: bool = True) -> int:
    return len(tokenizer.encode("" if pd.isna(text) else str(text), add_special_tokens=add_special_tokens, truncation=False))


def token_count_for_chunk(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=True, truncation=False))


def split_body_into_chunks(body: str, body_token_budget: int, overlap: int) -> list[dict]:
    """Разбиваю body на покрывающие токеновые окна с точным перекрытием."""
    if body_token_budget <= 0:
        raise ValueError("body_token_budget must be positive")
    if not (0 <= overlap < body_token_budget):
        raise ValueError("overlap must satisfy 0 <= overlap < body_token_budget")

    normalized_body = "" if body is None else str(body).strip()
    if not normalized_body:
        return []
    token_ids = tokenizer.encode(normalized_body, add_special_tokens=False, truncation=False)
    if not token_ids:
        return []

    chunks: list[dict] = []
    start = 0
    while start < len(token_ids):
        end = min(start + body_token_budget, len(token_ids))
        fragment = tokenizer.decode(token_ids[start:end], skip_special_tokens=True).strip()

        # В повторяющемся тексте соседние окна могут декодироваться одинаково
        # укорачиваю текущее окно, не нарушая перекрытие и покрытие.
        while (
            chunks and fragment == chunks[-1]["body_text"]
            and end - start > overlap + 1
        ):
            end -= 1
            fragment = tokenizer.decode(token_ids[start:end], skip_special_tokens=True).strip()

        if fragment and (not chunks or fragment != chunks[-1]["body_text"]):
            chunks.append({
                "body_text": fragment,
                "body_token_count": end - start,
                "token_start": start,
                "token_end": end,
            })
        elif end == len(token_ids):
            break

        if end >= len(token_ids):
            break
        next_start = end - overlap
        if next_start <= start:
            raise AssertionError("Chunker made no progress")
        start = next_start

    if chunks:
        assert chunks[0]["token_start"] == 0
        assert chunks[-1]["token_end"] == len(token_ids)
        for previous, current in zip(chunks, chunks[1:]):
            assert current["token_start"] <= previous["token_end"], "gap between chunks"
            assert previous["token_end"] - current["token_start"] == overlap, "unexpected overlap"
            assert current["body_text"] != previous["body_text"], "identical neighboring chunks"
    return chunks


def build_article_chunks(article_id: int, title: str, body: str, chunk_size: int, overlap: int) -> list[dict]:
    """Добавляю заголовочный префикс к каждому окну и слежу за общим лимитом токенов."""
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")

    title = "" if title is None else str(title).strip()
    prefix = f"Заголовок: {title}\n\nТекст статьи:\n"
    prefix_token_count = token_count_for_chunk(prefix)
    body_token_budget = chunk_size - prefix_token_count
    if body_token_budget <= 0:
        raise ValueError(
            f"Заголовочный префикс занимает {prefix_token_count} токенов и не помещается в chunk_size={chunk_size}"
        )
    if not (0 <= overlap < body_token_budget):
        raise ValueError(
            f"overlap={overlap} должен удовлетворять 0 <= overlap < body_token_budget={body_token_budget}"
        )

    body_without_duplicate_title = remove_duplicate_leading_title(title, "" if body is None else str(body))
    while True:
        body_chunks = split_body_into_chunks(body_without_duplicate_title, body_token_budget, overlap)
        if not body_chunks:
            body_chunks = [{"body_text": "", "body_token_count": 0, "token_start": 0, "token_end": 0}]
        assembled = []
        max_overflow = 0
        for body_chunk in body_chunks:
            chunk_text = prefix.rstrip() if not body_chunk["body_text"] else prefix + body_chunk["body_text"]
            token_count = token_count_for_chunk(chunk_text)
            max_overflow = max(max_overflow, token_count - chunk_size)
            assembled.append((chunk_text, token_count))
        if max_overflow <= 0:
            break
        body_token_budget -= max(1, max_overflow)
        if body_token_budget <= overlap:
            raise ValueError("После учёта префикса для body не осталось бюджета больше overlap")

    result = []
    chunk_count = len(assembled)
    for chunk_index, (chunk_text, token_count) in enumerate(assembled):
        if not chunk_text.strip():
            continue
        if result and result[-1]["chunk_text"] == chunk_text:
            continue
        result.append({
            "article_id": int(article_id),
            "chunk_index": chunk_index,
            "chunk_text": chunk_text,
            "token_count": int(token_count),
            "chunk_count": chunk_count,
        })
    for index, chunk in enumerate(result):
        chunk["chunk_index"] = index
        chunk["chunk_count"] = len(result)
        assert chunk["token_count"] <= chunk_size
        assert chunk["chunk_text"].startswith(f"Заголовок: {title}\n\nТекст статьи:")
        if index:
            assert chunk["chunk_text"] != result[index - 1]["chunk_text"]
    return result

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Токенизатор загружен: XLMRobertaTokenizerFast


In [10]:
from tqdm.auto import tqdm

articles_processed["document_token_count"] = articles_processed["document_text"].map(count_model_tokens)

print("Длина статей в токенах:")
print(articles_processed["document_token_count"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(0))

all_chunk_rows = []
for row in tqdm(list(articles_processed.itertuples(index=False)), desc="Чанкинг статей"):
    all_chunk_rows.extend(build_article_chunks(
        article_id=int(row.article_id),
        title=row.clean_title,
        body=row.clean_body,
        chunk_size=CHUNK_SIZE,
        overlap=CHUNK_OVERLAP,
    ))

article_chunks = pd.DataFrame(all_chunk_rows, columns=["article_id", "chunk_index", "chunk_text", "token_count", "chunk_count"])

assert article_chunks["article_id"].nunique() == len(articles_processed)
assert article_chunks["token_count"].le(CHUNK_SIZE).all()
assert article_chunks.groupby("article_id")["chunk_index"].apply(lambda s: s.tolist() == list(range(len(s)))).all()
assert article_chunks["chunk_text"].str.strip().ne("").all()

print(f"\nПостроено {len(article_chunks):,} чанков для {article_chunks['article_id'].nunique()} статей.")
print("Чанков на статью: медиана =", article_chunks.groupby("article_id").size().median(),
      ", максимум =", article_chunks.groupby("article_id").size().max())

Token indices sequence length is longer than the specified maximum sequence length for this model (167100 > 8192). Running this sequence through the model will result in indexing errors


Длина статей в токенах:
count       793.0
mean       1387.0
std        6160.0
min          18.0
50%         593.0
90%        2782.0
95%        4547.0
99%        7739.0
max      167100.0
Name: document_token_count, dtype: float64


Чанкинг статей:   0%|          | 0/793 [00:00<?, ?it/s]


Построено 3,787 чанков для 793 статей.
Чанков на статью: медиана = 2.0 , максимум = 545


Обращаю внимание на хвост распределения: есть одна экстремально длинная статья
(десятки тысяч токенов -> сотни чанков). Я её сознательно не обрезаю и не удаляю -
вместо этого на шаге 2 агрегация чанковых скоров устроена так, чтобы статья не получала
преимущество просто за счёт количества чанков.

## 7. Сохранение артефактов

Сохраняю на Диск четыре файла - с ними работают следующие ноутбуки. Исходные Feather-файлы
не изменяются.

In [11]:
articles_out = articles_processed[[
    "article_id", "raw_title", "raw_body", "clean_title", "clean_body",
    "document_text", "document_token_count",
]]
articles_out.to_parquet(ARTIFACTS_DIR / "articles_processed.parquet", index=False)

calibration_out = calibration[["query_id", "query_text", "ground_truth", "ground_truth_ids"]].copy()
calibration_out = calibration_out.rename(columns={"query_text": "raw_query_text"})
calibration_out["normalized_query_text"] = calibration_out["raw_query_text"].map(soft_normalize_text)
calibration_out.to_parquet(ARTIFACTS_DIR / "calibration_processed.parquet", index=False)

test_out = test[["query_id", "query_text"]].copy().rename(columns={"query_text": "raw_query_text"})
test_out["normalized_query_text"] = test_out["raw_query_text"].map(soft_normalize_text)
test_out.to_parquet(ARTIFACTS_DIR / "test_processed.parquet", index=False)

article_chunks[["article_id", "chunk_index", "chunk_count", "chunk_text", "token_count"]].to_parquet(
    ARTIFACTS_DIR / "article_chunks_384_58.parquet", index=False
)

for name in ["articles_processed.parquet", "calibration_processed.parquet", "test_processed.parquet", "article_chunks_384_58.parquet"]:
    path = ARTIFACTS_DIR / name
    print(f"Сохранено: {name} ({path.stat().st_size:,} байт)")

Сохранено: articles_processed.parquet (9,040,303 байт)
Сохранено: calibration_processed.parquet (74,280 байт)
Сохранено: test_processed.parquet (70,249 байт)
Сохранено: article_chunks_384_58.parquet (3,057,527 байт)


## Выводы

1. Данные чистые по структуре: идентификаторы уникальны, разметка парсится без аномалий.
2. У заметной доли запросов несколько релевантных статей - ранжировать нужно список, а не искать один ответ.
3. Запросы - живой пользовательский текст с опечатками и разговорными формулировками; статьи - канцелярский язык с HTML. Разрыв между этими стилями и есть главная сложность.
4. HTML очищен с сохранением видимого текста (списки, таблицы, вкладки); диагностика не нашла потерь.
5. Статьи разрезаны на чанки 384/58 с заголовком в каждом чанке; всё сохранено в `artifacts/` на Диске.

Дальше - шаг 2: поиск кандидатов (BM25 + векторный поиск) и честная схема валидации.